Installing Libraries:

In [ ]:
!pip install opencv-python
!pip install matplotlib
!pip install numpy
!pip install scikit-learn

Libraries:

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

Finding Height and Width of an Image:

In [ ]:
import cv2
import os

# Example sample image path from session 1
sample_image_path = '/content/drive/MyDrive/Datasets/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/ROIs/001/L_Fore/01.bmp'

# Load the image in grayscale
img = cv2.imread(sample_image_path, cv2.IMREAD_GRAYSCALE)

# Check if image was loaded successfully
if img is None:
    print("Image could not be loaded. Check the path.")
else:
    # Print its shape
    print("Image shape:", img.shape)

    # Print height and width
    height, width = img.shape
    print("Height:", height)
    print("Width:", width)


Preprocessing for Testing:

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

# === CONFIGURATION ===
BASE_PATH = r"/content/drive/MyDrive/Datasets/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/ROIs"
IMAGE_SIZE = (128, 60)
FINGER_LIST = ['L_Fore', 'L_Middle', 'L_Ring', 'R_Fore', 'R_Middle', 'R_Ring']
TRAIN_INDICES = [1, 2, 3, 4, 5, 6, 7]  # Protocol 1: First 7 images for training

# === STORAGE ===
train_images_2d = []
train_labels = []

# === STEP 1: LOAD INDIVIDUAL FINGER IMAGES (Strategy 2: No Fusion) ===
subject_dirs = sorted(os.listdir(BASE_PATH))
for subj in tqdm(subject_dirs, desc="Loading Protocol 1 - Strategy 2"):
    subject_path = os.path.join(BASE_PATH, subj)
    if not os.path.isdir(subject_path):
        continue

    for finger in FINGER_LIST:
        finger_path = os.path.join(subject_path, finger)
        for img_idx in TRAIN_INDICES:
            img_path = os.path.join(finger_path, f"{img_idx:02d}.bmp")
            print(f"📥 Loading: {img_path}")
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

            if img is None:
                print(f"❌ Missing: {img_path}")
                continue

            img = cv2.resize(img, IMAGE_SIZE)
            img_eq = exposure.equalize_hist(img).astype(np.float64)
            img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)

            train_images_2d.append(img_norm)
            label = f"{subj}_{finger}_img{img_idx:02d}"
            train_labels.append(label)
            print(f"✅ Added sample: {label}")

train_labels = np.array(train_labels)
print(f"\n✅ Total Training Samples: {len(train_images_2d)}")
print(f"✅ Example Image Shape: {train_images_2d[0].shape}")

# === STEP 2: COMPUTE 2DPCA ===
def compute_2dpca(images_2d, num_components):
    print("\n⚙️ Computing 2DPCA projection matrix...")
    n = len(images_2d)
    h, w = images_2d[0].shape
    mean_img = sum(images_2d) / n
    G_t = np.zeros((w, w))

    for i, img in enumerate(images_2d):
        A = img - mean_img
        G_t += A.T @ A
        if i < 3:
            print(f"  ➕ Sample {i+1} contribution added")

    G_t /= n
    eig_vals, eig_vecs = np.linalg.eigh(G_t)
    idx = np.argsort(-eig_vals)  # Descending order
    eig_vecs = eig_vecs[:, idx[:num_components]]
    print(f"✅ Projection matrix shape: {eig_vecs.shape}")
    return eig_vecs

num_components = 47  # Must be ≤ image width (60)
W = compute_2dpca(train_images_2d, num_components)

# === STEP 3: PROJECT INTO 2DPCA SPACE ===
projected_features = []
for i, img in enumerate(train_images_2d):
    feat = img @ W
    projected_features.append(feat)
    if i < 3:
        print(f"🧮 Projected shape of sample {i+1}: {feat.shape}")

# === STEP 4: FLATTEN FOR CLASSIFIER (e.g., kNN, SVM) ===
flat_features = np.array([feat.flatten() for feat in projected_features])
print(f"\n✅ Flattened feature matrix shape: {flat_features.shape}")


Testing:

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

# === CONFIGURATION ===
BASE_PATH = r"/content/drive/MyDrive/Datasets/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/ROIs"
IMAGE_SIZE = (128, 60)
FINGER_LIST = ['L_Fore', 'L_Middle', 'L_Ring', 'R_Fore', 'R_Middle', 'R_Ring']
TEST_INDICES = [8, 9, 10]  # Protocol 1: test images

test_images_2d = []
test_labels = []
test_paths = []

# === LOAD TEST DATA (Strategy 2: No Fusion) ===
subject_dirs = sorted(os.listdir(BASE_PATH))
for subj in tqdm(subject_dirs, desc="Loading Protocol 1 - Strategy 2 (Test Set)"):
    subject_path = os.path.join(BASE_PATH, subj)
    if not os.path.isdir(subject_path):
        continue

    for finger in FINGER_LIST:
        finger_path = os.path.join(subject_path, finger)
        for img_idx in TEST_INDICES:
            img_path = os.path.join(finger_path, f"{img_idx:02d}.bmp")
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

            if img is None:
                print(f"⚠️ Missing: {img_path}")
                continue

            print(f"✅ Using: {img_path}")
            img = cv2.resize(img, IMAGE_SIZE)
            img_eq = exposure.equalize_hist(img).astype(np.float64)
            img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)

            test_images_2d.append(img_norm)
            test_labels.append(f"{subj}_{finger}_img{img_idx:02d}")
            test_paths.append(img_path)

# === PROJECT TEST IMAGES INTO 2DPCA SPACE ===
proj_test_features = [img @ W for img in test_images_2d]
flat_test_features = np.array([f.flatten() for f in proj_test_features])
test_labels = np.array(test_labels)

print(f"\n✅ Projected test features shape: {flat_test_features.shape}")
print(f"📦 Total test samples: {len(flat_test_features)}")
print(f"🧾 Sample label: {test_labels[0]}")


Benchmarking:

In [ ]:
correct_matches = 0
total_tests = len(flat_test_features)

print("\n🔍 Starting classification using Manhattan distance...")

for i in range(total_tests):
    test_vec = flat_test_features[i]
    true_label = test_labels[i]  # e.g., "005_L_Fore_img08"

    # 📏 Compute Manhattan distance to all training vectors
    distances = np.sum(np.abs(flat_features - test_vec), axis=1)

    # 🏆 Find the closest match
    min_index = np.argmin(distances)
    predicted_label = train_labels[min_index]  # e.g., "005_L_Fore_img03"

    print(f"\nTest sample {i+1}:")
    print(f"  🎯 Predicted → {predicted_label}")
    print(f"  ✅ Actual    → {true_label}")

    # Compare subject ID and finger (first 2 parts of the label)
    pred_subject, pred_finger = predicted_label.split('_')[0], predicted_label.split('_')[1]
    true_subject, true_finger = true_label.split('_')[0], true_label.split('_')[1]

    if pred_subject == true_subject and pred_finger == true_finger:
        correct_matches += 1
        print("  🟢 Match (Subject + Finger correct)")
    else:
        print("  🔴 Mismatch")

# 📈 Compute final accuracy
accuracy = (correct_matches / total_tests) * 100
print(f"\n🏁 Final recognition accuracy: {accuracy:.2f}% ({correct_matches}/{total_tests})")
